<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Gradient Descent and Step Size

**Use slopes to improve clock settings and identify when the step size prevents convergence.**

Retain the clock observations and squared-error objective. Remove calibration and the correction bound; first fix the rate to zero, then vary both settings.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · From a slope to an update

The fixed errors $d_i$ are $-2,-1,-1,0$ min at times $t_i=0,1,2,3$ h, respectively, for $i=1,\ldots,4$. Choose initial correction $q$ (min) and rate correction $r$ (min/h). The remaining errors and squared-error objective are

$$
y_i=d_i+q+rt_i,\qquad \phi(q,r)=\sum_{i=1}^4 y_i^2.
$$

Equations below use numerical values in these stated units.

With $r=0$, write $f(q)=\phi(q,0)=4(q-1)^2+2$, so $f'(q)=8(q-1)$. Gradient descent uses

$$
q^{(k+1)}=q^{(k)}-\alpha f'(q^{(k)}).
$$

The superscript $k$ counts numerical candidates, not physical time. The positive step size $\alpha$ is a hyperparameter: it changes the search, not the errors of a fixed decision.

Starting at $q^{(0)}=0$ with $\alpha=0.05$ gives $q^{(1)}=0.4$, then $q^{(2)}=0.64$. Scores fall from $6$ to $3.44$ to $2.5184$ min²; the exact minimum is $2$ at $q=1$. The first arrow follows the negative slope.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/offset_descent.svg" alt="A negative-gradient update increases correction from zero to 0.4 and lowers squared error." width="520" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Why the step size matters

The error relative to the optimum obeys

$$
q^{(k+1)}-1=(1-8\alpha)(q^{(k)}-1).
$$

It shrinks when $|1-8\alpha|<1$, or $0<\alpha<0.25$. This range is specific to this one-variable quadratic.

| $\alpha$ | First settings from $0$ | Behavior |
|:---|:---|:---|
| $0.05$ | $0,\ 0.4,\ 0.64$ | Converges from below |
| $0.125$ | $0,\ 1,\ 1$ | Reaches the minimum in one step |
| $0.20$ | $0,\ 1.6,\ 0.64$ | Alternates while converging |
| $0.25$ | $0,\ 2,\ 0$ | Oscillates without convergence |
| $0.30$ | $0,\ 2.4,\ -0.96$ | Diverges |

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Update both settings together

Now release $r$ and write $x=(q,r)$, $f(x)=\phi(q,r)$. Both settings are unrestricted:

$$
\nabla f=\begin{bmatrix}8q+12r-8\\12q+28r-6\end{bmatrix},\qquad
x^{(k+1)}=x^{(k)}-\alpha\nabla f(x^{(k)}).
$$

Both components use the old candidate. From $(0,0)$, $\alpha=0.05$ gives $(0.4,0.3)$, then $(0.46,-0.06)$, with scores $4.34$ and $3.2456$ min². The negative gradient has directional slope $-\|\nabla f\|_2^2<0$ away from stationarity, but only a sufficiently small finite step is guaranteed to decrease the score.

The one-variable step-size range no longer applies because the settings are coupled. For example, $\alpha=0.05$ converges here, while $0.10$ diverges. The exact reference is $(1.9,-0.6)$ with score $0.2$ min². Equal-score contours reveal the zigzag path.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/gradient_update_path.svg" alt="The two-setting path zigzags toward the unrestricted minimum because the settings are coupled." width="520" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Interactive comparison

The “settings” slider selects one setting ($r=0$) or two unrestricted settings. Hold that choice fixed to compare step sizes $\alpha$. Each trace starts at zero and shows 12 updates; the vertical axis is logarithmic. The reference score changes from $2$ to $0.2$ when the rate is released. A fixed update count is not a convergence declaration.

In [ ]:
import sys
import numpy as np
import matplotlib

if sys.platform != "emscripten":
    try:
        matplotlib.use("widget", force=True)
    except (RuntimeError, ValueError):
        from matplotlib.backends import backend_registry
        backend_registry._clear()
        matplotlib.use("widget", force=True)
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

BLUE, TEAL, ORANGE, PURPLE, GRAY = "#2878b5", "#168578", "#e78b24", "#8854a5", "#707070"
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11})

# Fixed observations and the physical response.
observed_errors = np.array([-2., -1., -1., 0.])  # min
observation_times = np.arange(4.)  # h

def evaluate_clock(decision):
    correction, rate = decision
    errors = observed_errors + correction + rate * observation_times
    gradient = 2 * np.array([errors.sum(), observation_times @ errors])
    return {"decision": np.array(decision), "errors": errors,
            "objective": float(errors @ errors), "gradient": gradient}

def gradient_trace(step_size, setting_count, steps=12):
    decision = np.zeros(2)
    history = [evaluate_clock(decision)]
    for _ in range(steps):
        gradient = history[-1]["gradient"].copy()
        if setting_count == 1:
            gradient[1] = 0  # Hold the rate at zero.
        decision = decision - step_size*gradient
        history.append(evaluate_clock(decision))
    return history

In [ ]:
figure, axis = plt.subplots(figsize=(5.6, 4.8))
figure.subplots_adjust(left=0.16, right=0.96, top=0.90, bottom=0.41)
path, = axis.plot([], [], "o-", color=BLUE, markersize=4)
reference = axis.axhline(2, color=TEAL, linestyle="--", label="Exact minimum")
axis.set(xlabel="Update k", ylabel="Squared error (min², log scale)", yscale="log",
         title="Step size changes convergence", xticks=[0, 3, 6, 9, 12])
axis.grid(alpha=0.25)
axis.legend(loc="upper left", fontsize=10)
status = figure.text(0.16, 0.25, "", va="top")
alpha_slider = Slider(figure.add_axes([0.25, 0.12, 0.58, 0.035]), "alpha",
                      0.005, 0.3, valinit=0.05, valstep=0.005, color=PURPLE)
settings_slider = Slider(figure.add_axes([0.25, 0.05, 0.58, 0.035]), "settings",
                         1, 2, valinit=1, valstep=1, valfmt="%d", color=BLUE)

def update(_):
    count = int(settings_slider.val)
    history = gradient_trace(alpha_slider.val, count)
    scores = [record["objective"] for record in history]
    exact = 2 if count == 1 else 0.2
    path.set_data(range(len(scores)), scores)
    reference.set_ydata([exact, exact])
    axis.set_xlim(-0.3, 12.3)
    axis.set_ylim(exact*0.7, max(scores)*2)
    q, r = history[-1]["decision"]
    status.set_text(f"After 12 updates: q = {q:.3g} min; r = {r:.3g} min/h\n"
                    f"Score = {scores[-1]:.4g} min²; reference = {exact:g} min²")
    figure.canvas.draw_idle()

figure._sliders = [alpha_slider, settings_slider]
for slider in figure._sliders:
    slider.on_changed(update)
update(None)
plt.show()